In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 100)

# Load the raw listings 
raw_listings_path = "../data/raw/listings.csv.gz"
df_raw = pd.read_csv(raw_listings_path, compression='gzip', low_memory=False)

print(f"Listings Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")


Listings Dataset Shape: 30,259 rows x 90 columns


In [5]:
target_cols = [
    'id', 'name', 'price', 'bathrooms_text', 'bedrooms', 'beds', 
    'amenities', 'latitude', 'longitude', 'host_is_superhost', 
    'number_of_reviews', 'review_scores_rating'
]

available_cols = [c for c in target_cols if c in df_raw.columns]
display(df_raw[available_cols].dtypes.to_frame("Data Type"))
display(df_raw[available_cols].head(3))

,Data Type
id,int64
name,str
price,str
bathrooms_text,str
bedrooms,float64
beds,float64
amenities,str
latitude,float64
longitude,float64
host_is_superhost,str


,id,name,price,bathrooms_text,bedrooms,beds,amenities,latitude,longitude,host_is_superhost,number_of_reviews,review_scores_rating
0,2539,11 Min to Manhattan • Prospect Park • Fast WiFi,$113.97,1 shared bath,1.0,2.0,"[""Bed linens"", ""Hot water"", ""Exterior security...",40.645937,-73.972164,f,10,4.8
1,6848,Only 2 stops to Manhattan studio,$117.27,1 bath,2.0,1.0,"[""Bed linens"", ""Hot water"", ""Fire extinguisher...",40.709350,-73.953420,t,198,4.6
2,6872,Uptown Sanctuary w/ Private Bath (Month to Month),$80.06,1 shared bath,NaN,1.0,"[""Dryer"", ""Exterior security cameras on proper...",40.801070,-73.942550,f,2,5.0


In [8]:
print("=== DIRTY DATA AUDIT REPORT ===")

# Price formatting ($ and commas)
if df_raw['price'].dtype == object:
    dollar_count = df_raw['price'].str.contains(r'\$', na=False).sum()
    comma_count = df_raw['price'].str.contains(r',', na=False).sum()
    print(f"[!] Price: {dollar_count:,} entries have '$' and {comma_count:,} have commas.")

# Amenities string format
sample_amenity = df_raw['amenities'].dropna().iloc[0]
print(f"[!] Amenities: Stored as type '{type(sample_amenity).__name__}'. Sample:\n    {sample_amenity[:75]}...")

# Bathrooms messy text
if 'bathrooms_text' in df_raw.columns:
    print(f"[!] Bathrooms Text: Samples of text representations:\n    {df_raw['bathrooms_text'].dropna().unique()[:6].tolist()}")

#  Host boolean flags
if 'host_is_superhost' in df_raw.columns:
    print(f"[!] Superhost Flag counts:\n{df_raw['host_is_superhost'].value_counts(dropna=False)}")

# Coordinate checks (Bounds check for NYC: Lat 40.4-40.9, Lon -74.3 to -73.7)
lat_outliers = ~df_raw['latitude'].between(40.4, 41.0)
lon_outliers = ~df_raw['longitude'].between(-74.3, -73.6)
print(f"[!] Coordinate Outliers: {lat_outliers.sum()} lat outliers, {lon_outliers.sum()} lon outliers.")


=== DIRTY DATA AUDIT REPORT ===
[!] Amenities: Stored as type 'str'. Sample:
    ["Bed linens", "Hot water", "Exterior security cameras on property", "Clean...
[!] Bathrooms Text: Samples of text representations:
    ['1 shared bath', '1 bath', '1 private bath', '2.5 baths', '2 baths', '1.5 baths']
[!] Superhost Flag counts:
host_is_superhost
f      22814
t       7096
NaN      349
Name: count, dtype: int64
[!] Coordinate Outliers: 0 lat outliers, 0 lon outliers.


In [7]:
# Subway stations check
df_subway = pd.read_csv("../data/raw/subway_stations.csv")
print(f"Subway Stations Loaded: {len(df_subway):,} stations")
display(df_subway.head(2))

# Neighborhoods GeoJSON check
gdf_neighborhoods = gpd.read_file("../data/raw/neighbourhoods.geojson")
print(f"Neighborhood Polygons Loaded: {len(gdf_neighborhoods):,} areas")
display(gdf_neighborhoods.head(2))


Subway Stations Loaded: 496 stations


,GTFS Stop ID,Station ID,Complex ID,Division,Line,Stop Name,Borough,CBD,Daytime Routes,Structure,GTFS Latitude,GTFS Longitude,North Direction Label,South Direction Label,ADA,ADA Northbound,ADA Southbound,ADA Notes,Georeference
0,R43,37,37,BMT,4th Av,77 St,Bk,False,R,Subway,40.629742,-74.025510,Manhattan,Bay Ridge,0,0,0,NaN,POINT (-74.02551 40.629742)
1,M01,108,108,BMT,Myrtle Av,Middle Village-Metropolitan Av,Q,False,M,Elevated,40.711396,-73.889601,Inbound,Last Stop,1,1,1,NaN,POINT (-73.889601 40.711396)


Neighborhood Polygons Loaded: 233 areas


,neighbourhood,neighbourhood_group,geometry
0,Bayswater,Queens,"MULTIPOLYGON (((-73.76671 40.61491, -73.76825 ..."
1,Allerton,Bronx,"MULTIPOLYGON (((-73.8486 40.87167, -73.84582 4..."
